# DAMPS-MMHCL Training for Amazon Clothing Dataset (Local)

This notebook trains the **DAMPS-MMHCL** (Spectral Domain Representation Calibration upgrade for the Multi-Modal Hypergraph Contrastive Learning framework) on the Amazon Clothing dataset on a local machine with NVIDIA RTX 5090 (32 GB VRAM).

It is a drop-in replacement for the original MMHCL notebook and follows the *Revision 11 — Phase 1 (Quick Win) Execution Plan with Quantitative Stop-Gate* architecture spec (see `DAMPS_to_MMHCL_architecture_revision44.tex` / `.pdf`), which builds on the *Revision 9 — 100% Compliance Check & Final Lock* baseline (`DAMPS_to_MMHCL_architecture_revision42.tex` / `.pdf`).

**Phase 1 — Quick Win:** the notebook now sweeps four configurations × **5 seeds** (20 training runs total) to break the *τ-saturation embedding-collapse* failure mode documented in rev44 §3 (10.7 % asymmetric Recall@20 deficit on Amazon Clothing):

| Variant | Flags | Hypothesis |
| ------- | ----- | ---------- |
| (a) anchor   | `--temperature 0.1 --learnable_tau 1 --damps_avrf 1` | rev42 baseline; τ saturates at ~0.0909 |
| (b) tau03    | `--temperature 0.3 --learnable_tau 0 --damps_avrf 1` | static τ removes saturation |
| (c) avrf_off | `--temperature 0.1 --learnable_tau 1 --damps_avrf 0` | AVRF off recovers sparse coverage |
| (d) combined | `--temperature 0.3 --learnable_tau 0 --damps_avrf 0` | **RECOMMENDED** combined fix |

The Phase 1 stop-gate (rev44 §5.2): `Recall@20 ≥ 0.0870 ∧ NDCG@20 ≥ 0.0390` (mean over the **5 seeds** per variant below) with paired *t*-test `p < 0.05` vs. the rev42 anchor. `Recall@20 ≥ 0.0900` fully validates the paper contribution. (Increase `n_runs` in the training cell if you need a stricter 10-seed protocol.)

## DAMPS pipeline (per forward pass)
1. **Spectral decomposition** — 1-D rFFT on the projected modality features (d = 64).
2. **Metadata-Aware APC** — von-Mises MLE phase calibration grouped by static metadata categories (no k-means).
3. **AVRF** — logit-clipped Wiener gate with strict `[-2, 2]` initialisation prior (data-driven from raw modality SNR) and per-epoch EMA MAD aggregation.
4. **Residual IMCF** — ASC consensus coefficient applied in residual form to avoid multiplicative attenuation.
5. **Inverse FFT** — back to spatial domain → fed into HGCN via **Soft Residual-Routing** (eq. 3 of the spec).

Auxiliary upgrades wired into the trainer:
- **Pattern B' (Scheduled Rebuild)** — reconstruct the K-NN multi-modal hypergraph every `R` epochs from the **Slim Momentum** buffers (98 % VRAM saving vs MoCo).
- **Dual-path K-NN** — chunked PyTorch (default) with FAISS HNSW fallback when N ≥ 60 000.
- **`bfloat16` mixed precision** (-30 / -40 % wall-clock, -40 % VRAM on Ada Lovelace).
- **Static InfoNCE temperature τ = 0.3** (rev44 Phase 1 default; toggle back to learnable nn.Parameter via `--learnable_tau 1`).
- **cuFFT plan-cache lockdown** (`cufft_plan_cache.max_size = 1`, Section 3.3).
- **Diagnostic logs** — NNZ / avg_deg per rebuild, tanh-saturation rate of the AVRF gates.

## Configuration
- **GPU**: NVIDIA RTX 5090 (32 GB VRAM)
- **Working dir**: `MMHCL_DAMPS_Project/`
- **Entry point**: `train.py` (replaces the original `codes/main.py`)
- **Batch Size**: 1024
- **Max Epochs**: 250
- **Rebuild cadence**: every 5 epochs after a 10-epoch warm-up
- **W&B Project**: `damps-mmhcl-clothing`

## Steps
1. Verify environment and GPU
2. Install dependencies for the DAMPS-MMHCL package
3. Set up W&B logging
4. (Optional) GPU memory monitor
5. Run a quick smoke test of the DAMPS package
6. **rev44 Phase 1 Item #1** — 7-point eval-protocol audit (must PASS before training)
7. **Root-cause roadmap §5.0** — Day 1 diagnostic sprint (D1 on CPU; optional D2/D3/APC probe on GPU via `MMHCL_DAMPS_Project/scripts/day1_diagnostic_sprint.py`; notebook §3.0.5)
8. Multi-seed training of the full DAMPS-MMHCL model — **4 variants × 5 seeds** (20 runs)
9. Aggregate results per variant
10. **rev44 Phase 1** — Bonferroni-corrected paired *t*-tests vs. anchor + stop-gate verdict
11. Export results to Excel

## 1. Environment Setup & GPU Verification

In [1]:
from __future__ import annotations

import os
import sys
import torch

# ---------------------------------------------------------------------------
#  Resolve project root.  The notebook may be opened from either the
#  workspace root or one of the package sub-directories.
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(('codes', 'MMHCL_DAMPS_Project')):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

# DAMPS-MMHCL implementation lives under MMHCL_DAMPS_Project/
DAMPS_DIR: str = os.path.join(PROJECT_ROOT, 'MMHCL_DAMPS_Project')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')
# Legacy CODES_DIR is still around for the original MMHCL baseline ablation
CODES_DIR: str = os.path.join(PROJECT_ROOT, 'codes')

print(f"Project Root        : {PROJECT_ROOT}")
print(f"DAMPS-MMHCL Directory: {DAMPS_DIR}")
print(f"Data Directory       : {DATA_DIR}")

assert os.path.exists(DAMPS_DIR), (
    f"MMHCL_DAMPS_Project not found at {DAMPS_DIR}. "
    f"Make sure you cloned the full DAMPS-MMHCL repository."
)
assert os.path.exists(DATA_DIR), f"Data directory not found: {DATA_DIR}"
print("\n[OK] All directories verified")

# Verify GPU & bfloat16 capability (DAMPS uses bf16 AMP by default)
print("\n" + "=" * 60)
print("GPU Information")
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    total_gb: float = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU Memory  : {total_gb:.1f} GB")

    # bfloat16 is a hard requirement for the DAMPS bf16 AMP path. Detect
    # support so the user gets a clear error early instead of a cryptic
    # autocast failure mid-training.
    bf16_ok: bool = torch.cuda.is_bf16_supported()
    print(f"bfloat16 support: {bf16_ok}")
    if not bf16_ok:
        print(
            "[WARNING] bfloat16 not supported on this GPU. Pass --use_amp 0 "
            "to fall back to fp32, or run on Ampere/Ada Lovelace+."
        )
else:
    print("[WARNING] No GPU detected. DAMPS training will be very slow on CPU.")

Project Root        : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing
DAMPS-MMHCL Directory: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
Data Directory       : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data

[OK] All directories verified

GPU Information
CUDA available: True
GPU         : NVIDIA GeForce RTX 5090
CUDA version: 12.8
GPU Memory  : 31.8 GB
bfloat16 support: True


## 1.5 Install Dependencies (Run Once)

In [2]:
# Install required dependencies for the DAMPS-MMHCL package (run once).
#
# The DAMPS-MMHCL implementation pins its requirements in
# `MMHCL_DAMPS_Project/requirements.txt`. We install from there first,
# then add notebook-only extras (wandb, pandas, openpyxl).
from __future__ import annotations

import subprocess
import sys
import os

print("=" * 60)
print("Installing DAMPS-MMHCL dependencies")
print("=" * 60)

# UV-managed Python sometimes refuses installs without --break-system-packages
pip_extra_args: list[str] = []
test_result: subprocess.CompletedProcess[str] = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', 'pip'],
    capture_output=True, text=True,
)
if 'externally-managed' in test_result.stderr:
    pip_extra_args = ['--break-system-packages']
    print("\n[INFO] UV-managed Python detected, using --break-system-packages")

# Step 1 - install DAMPS-MMHCL package requirements
requirements_path: str = os.path.join(DAMPS_DIR, 'requirements.txt')
if not os.path.exists(requirements_path):
    requirements_path = os.path.join(PROJECT_ROOT, 'requirements.txt')

if os.path.exists(requirements_path):
    print(f"\n1. Installing packages from: {requirements_path}")
    print("   This may take several minutes for PyTorch (~2.6 GB)...")
    cmd: list[str] = [sys.executable, '-m', 'pip', 'install', '-r', requirements_path] + pip_extra_args
    result: subprocess.CompletedProcess[str] = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"   [ERROR] Installation failed! Exit code: {result.returncode}")
        print("\n--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise Exception("Failed to install requirements")
    print("   [OK] DAMPS-MMHCL package requirements installed")
else:
    print(f"   [WARNING] No requirements.txt found at {requirements_path}")

# Step 2 - install notebook-only extras
additional_packages: list[str] = ['wandb', 'openpyxl', 'pandas']
print(f"\n2. Installing notebook extras: {additional_packages}")
for package in additional_packages:
    print(f"   Installing {package}...")
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', package] + pip_extra_args
    subprocess.run(cmd, capture_output=True)
print("   [OK] Notebook extras installed")

# Step 3 - sanity-check imports of all DAMPS-MMHCL modules
print("\n3. Verifying DAMPS-MMHCL package imports...")
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)
try:
    import damps  # noqa: F401
    from damps import (  # noqa: F401
        DAMPS,
        SlimMomentumEncoder,
        DualPathKNN,
        compute_avrf_prior,
        compute_avrf_logit,
    )
    print("   [OK] `damps` core package imports cleanly")
except Exception as exc:  # pragma: no cover
    print(f"   [WARNING] Could not import `damps` package yet: {exc}")
    print("   This is expected on the very first run; restart the kernel and try again.")

print("\n" + "=" * 60)
print("[OK] All packages installed successfully!")
print("=" * 60)
print("\n[IMPORTANT] If this is the first install, RESTART THE KERNEL before running the next cells.")

Installing DAMPS-MMHCL dependencies

1. Installing packages from: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\requirements.txt
   This may take several minutes for PyTorch (~2.6 GB)...
   [OK] DAMPS-MMHCL package requirements installed

2. Installing notebook extras: ['wandb', 'openpyxl', 'pandas']
   Installing wandb...
   Installing openpyxl...
   Installing pandas...
   [OK] Notebook extras installed

3. Verifying DAMPS-MMHCL package imports...
   [OK] `damps` core package imports cleanly

[OK] All packages installed successfully!

[IMPORTANT] If this is the first install, RESTART THE KERNEL before running the next cells.


## 2. Weights & Biases Setup

In [1]:
from __future__ import annotations

import wandb
from IPython.display import display, Markdown

# ========== Login to Weights & Biases ==========
# A dedicated project so DAMPS runs do not get mixed in with the original
# MMHCL baseline runs. Change `wandb_entity` if you push to your own account.
wandb_project: str = 'damps-mmhcl-clothing'
wandb_entity: str = 'baitapck51cc-uet'

# Your W&B API key (replace with your own if you fork this notebook)
WANDB_API_KEY: str = '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'

wandb.login(key=WANDB_API_KEY)

print("[OK] W&B login successful")
print(f"     Entity : {wandb_entity}")
print(f"     Project: {wandb_project}")

wandb_url: str = f"https://wandb.ai/{wandb_entity}/{wandb_project}"
print()
display(Markdown(f"## [Open Weights & Biases Dashboard]({wandb_url})"))
print(f"Direct Link: {wandb_url}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Anh Khoi\_netrc
wandb: Currently logged in as: baitapck51cc (baitapck51cc-uet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[OK] W&B login successful
     Entity : baitapck51cc-uet
     Project: damps-mmhcl-clothing



## [Open Weights & Biases Dashboard](https://wandb.ai/baitapck51cc-uet/damps-mmhcl-clothing)

Direct Link: https://wandb.ai/baitapck51cc-uet/damps-mmhcl-clothing


## 2.5 GPU Memory Monitor (Optional)

Start a lightweight background monitor to print GPU VRAM usage while training.
Run the stop function after training completes.

In [2]:
from __future__ import annotations

import threading
import time
import torch
import csv
import os
from typing import Optional

# GPU monitor controls
GPU_MONITOR_RUNNING: bool = False
GPU_MONITOR_THREAD: Optional[threading.Thread] = None


def _resolve_monitor_root() -> str:
    current_dir: str = os.getcwd()
    if current_dir.endswith('codes'):
        return os.path.dirname(current_dir)
    return current_dir


def _gpu_monitor(interval_seconds: int, csv_path: str, print_live: bool) -> None:
    global GPU_MONITOR_RUNNING
    if not torch.cuda.is_available():
        print("[WARNING] CUDA not available. GPU monitor will not run.")
        GPU_MONITOR_RUNNING = False
        return

    total_mem: int = torch.cuda.get_device_properties(0).total_memory
    start_time: float = time.time()

    file_exists: bool = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as csv_file:
        writer: csv.writer = csv.writer(csv_file)
        if not file_exists:
            writer.writerow([
                'relative_time_s',
                'allocated_gb',
                'reserved_gb',
                'allocated_pct',
                'reserved_pct'
            ])

        print(f"[INFO] GPU monitor started. Logging to: {csv_path}")
        while GPU_MONITOR_RUNNING:
            allocated: int = torch.cuda.memory_allocated(0)
            reserved: int = torch.cuda.memory_reserved(0)
            allocated_pct: float = (allocated / total_mem) * 100
            reserved_pct: float = (reserved / total_mem) * 100
            rel_time: float = time.time() - start_time

            writer.writerow([
                f"{rel_time:.2f}",
                f"{allocated / 1024**3:.4f}",
                f"{reserved / 1024**3:.4f}",
                f"{allocated_pct:.2f}",
                f"{reserved_pct:.2f}",
            ])
            csv_file.flush()

            if print_live:
                print(
                    f"[GPU] allocated={allocated/1024**3:.2f} GB ({allocated_pct:.1f}%), "
                    f"reserved={reserved/1024**3:.2f} GB ({reserved_pct:.1f}%)"
                )

            time.sleep(interval_seconds)

    print("[INFO] GPU monitor stopped.")


def start_gpu_monitor(interval_seconds: int = 10, csv_filename: str = 'gpu_memory_monitor.csv', print_live: bool = False) -> None:
    """Start background GPU monitor. Logs to CSV every N seconds."""
    global GPU_MONITOR_RUNNING, GPU_MONITOR_THREAD
    if GPU_MONITOR_RUNNING:
        print("[INFO] GPU monitor already running.")
        return

    monitor_root: str = _resolve_monitor_root()
    csv_path: str = os.path.join(monitor_root, csv_filename)

    GPU_MONITOR_RUNNING = True
    GPU_MONITOR_THREAD = threading.Thread(
        target=_gpu_monitor,
        args=(interval_seconds, csv_path, print_live),
        daemon=True
    )
    GPU_MONITOR_THREAD.start()


def stop_gpu_monitor() -> None:
    """Stop background GPU monitor."""
    global GPU_MONITOR_RUNNING
    GPU_MONITOR_RUNNING = False


# Start monitor with 15s interval (adjust as needed)
# Set print_live=True if you want console output too
start_gpu_monitor(interval_seconds=15, csv_filename='gpu_memory_monitor.csv', print_live=False)

# Call stop_gpu_monitor() after training completes.

[INFO] GPU monitor started. Logging to: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\gpu_memory_monitor.csv


## 2.6 DAMPS Smoke Test (Optional)

Run a small end-to-end sanity check of the DAMPS-MMHCL package before launching the full multi-seed training. This invokes `MMHCL_DAMPS_Project/tests/smoke_test.py`, which exercises:

- AVRF prior derivation + logit clipping
- DAMPS core forward pass (FFT → APC → AVRF → Residual IMCF → iFFT)
- Slim Momentum Encoder update step
- Dual-Path KNN graph construction
- Full `DAMPS_MMHCL` model forward + backward

Skip this cell if you have already validated the package.

In [3]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

# ---------------------------------------------------------------------------
# Make this cell self-sufficient: section 2.6 is documented as "Optional",
# so users often run it without first executing the Section 2.1 env-setup
# cell that defines DAMPS_DIR. If DAMPS_DIR is not yet in the namespace,
# auto-discover it from the notebook's working directory using three
# resolution strategies, then publish it back so the rest of the cell (and
# any downstream cells executed in this same kernel) can rely on it.
# ---------------------------------------------------------------------------
try:
    _damps_dir: str = DAMPS_DIR  # populated by the env-setup cell
except NameError:
    _cwd = Path.cwd().resolve()
    if _cwd.name == 'MMHCL_DAMPS_Project':
        _damps_dir = str(_cwd)
    elif (_cwd / 'MMHCL_DAMPS_Project').is_dir():
        _damps_dir = str((_cwd / 'MMHCL_DAMPS_Project').resolve())
    else:
        _ancestor = next(
            (p for p in [_cwd, *_cwd.parents]
             if (p / 'MMHCL_DAMPS_Project').is_dir()),
            None,
        )
        if _ancestor is None:
            raise RuntimeError(
                "Could not locate the MMHCL_DAMPS_Project directory.\n"
                "Either run the environment-setup cell (Section 2.1) first, "
                "or open this notebook from the repository root."
            )
        _damps_dir = str((_ancestor / 'MMHCL_DAMPS_Project').resolve())
    DAMPS_DIR = _damps_dir
    print(f"[smoke-test] DAMPS_DIR auto-resolved to: {_damps_dir}")

smoke_path: str = os.path.join(DAMPS_DIR, 'tests', 'smoke_test.py')

if not os.path.exists(smoke_path):
    print(f"[WARN] Smoke test not found at {smoke_path} - skipping.")
else:
    print("=" * 70)
    print("Running DAMPS-MMHCL smoke test...")
    print(f"  Script: {smoke_path}")
    print("=" * 70)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    env['PYTHONUNBUFFERED'] = '1'

    result = subprocess.run(
        [sys.executable, smoke_path],
        cwd=DAMPS_DIR,
        capture_output=True,
        text=True,
        env=env,
        encoding='utf-8',
        errors='replace',
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("\n[FAIL] Smoke test exited with code", result.returncode)
        if result.stderr:
            print("\n--- STDERR ---")
            print(result.stderr)
    else:
        print("\n[OK] DAMPS-MMHCL smoke test passed.")

[smoke-test] DAMPS_DIR auto-resolved to: C:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
Running DAMPS-MMHCL smoke test...
  Script: C:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\tests\smoke_test.py
== DAMPS core ==
  [OK] forward + backward, 133 params
  [OK] tanh_sat: {'img': 0.0, 'txt': 0.0, 'aud': 0.0}
  [OK] update_epoch_mad runs
  [OK] IMCF schedule: forward-pass counter +5, epoch held at 3 then advanced to 7 via set_epoch
== Slim Momentum Encoder ==
  [OK] EMA buffers update + initialised flag
== Dual-path KNN ==
  [OK] single-modality K-NN, NNZ=256
  [OK] multi-modal hypergraph, NNZ=1594, avg_deg=24.91
== Data-driven prior ==
  [OK] prior range=[0.401,0.693]  logit clipped
== DAMPS_MMHCL full model ==
  [OK] static tau initialised at 0.3000 (rev44 Phase 1 anchor=0.3, registered as buffer)
  [OK] forward pass shapes OK
  [OK] backw

## 3. Multi-Seed Training (DAMPS-MMHCL) — rev44 Phase 1 Four-Variant Sweep

Train the **DAMPS-MMHCL** model across the **four rev44 / Revision 11 Phase 1 variants × 5 random seeds** = **20** training runs. The cell calls `MMHCL_DAMPS_Project/train.py` directly — the same entry-point used by `scripts/run_phase1_ablation.py`. (To match a 10-seed paper protocol, set `n_runs = 10` in the next code cell for 40 runs total.)

After the eval-protocol audit (§3.0), you can run **§3.0.5** *before* this sweep: D1 (CPU) plus optional D2/D3/APC GPU probes from `scripts/day1_diagnostic_sprint.py` (see `Phase1_RootCause_Analysis_and_Remediation_Roadmap.tex` and `MMHCL_DAMPS_Project/README.md` §5.0).

### Backbone hyperparameters (parity with original MMHCL paper)
- **Batch Size**: 1024 (RTX 5090, 32 GB VRAM)
- **Max Epochs**: 250
- **Learning Rate**: 0.0001
- **Embedding Dim**: 64 / **Top-k**: 5
- **τ**: variant-dependent (anchor 0.1 learnable, Phase 1 0.3 static)
- **`User_layers` / `Item_layers`**: 3 / 2
- **`user_loss_ratio` / `item_loss_ratio`**: 0.03 / 0.07
- **Early Stopping**: patience = 30, min_epochs = 75, monitor = `val_recall@20`
- **ReduceLROnPlateau**: factor = 0.5, patience = 3

### Per-variant overrides (rev44 §4)

| Variant | `--temperature` | `--learnable_tau` | `--damps_avrf` |
| ------- | --------------- | ----------------- | -------------- |
| `anchor`   (a) — rev42 baseline | 0.1 | 1 | 1 |
| `tau03`    (b) — static τ only  | 0.3 | 0 | 1 |
| `avrf_off` (c) — AVRF off only  | 0.1 | 1 | 0 |
| `combined` (d) — recommended    | 0.3 | 0 | 0 |

### Other DAMPS-specific flags (rev44, kept constant across variants)
- `--damps_apc 1` — Metadata-Aware Adaptive Phase Calibration
- `--damps_imcf 1` — Residual Inter-Modal Coherence Filter
- `--damps_soft_routing 1` — Soft Residual-Routing into HGCN
- `--damps_momentum 1` — Slim Momentum Encoder
- `--damps_data_driven_prior 1` — SNR-derived AVRF priors
- `--damps_num_categories 10` — # static metadata clusters for APC
- `--damps_warmup_epochs 10` — adaptive EMA warm-up
- `--rebuild_R 5` — Pattern B' rebuild cadence
- `--knn_chunk_size 4096` / `--faiss_threshold 60000` — Dual-path K-NN
- `--use_amp 1` — bfloat16 mixed precision
- `--clip_grad_norm 1.0` — gradient clipping

### Subset to a single variant (optional)
To run **only** the recommended Phase 1 config (variant `combined`), set `variants_to_run = {'combined': phase1_variants['combined']}` near the top of the next code cell. The default value `variants_to_run = phase1_variants` runs all four (**20 runs** total with `n_runs = 5`).

### 3.0 rev44 Phase 1 Item #1 — 7-point Eval Protocol Audit (mandatory pre-flight)

Per `DAMPS_to_MMHCL_architecture_revision44.tex` Section 4 / `Phase1_Fixed_Implementation_Audit_rev44.tex` Finding 3, this audit **must PASS** before launching the four-variant Phase 1 sweep below. It runs `MMHCL_DAMPS_Project/scripts/audit_eval_protocol.py`, which verifies all seven checklist items against the on-disk Clothing dataset:

| # | Check | Source of truth |
| - | ----- | --------------- |
| (i)   | 5-core filtering threshold              | `args.core` |
| (ii)  | Train/val/test split + reproducibility  | counts in the JSON splits |
| (iii) | All-ranking vs sampled evaluation       | static analysis of `utility/batch_test.py::test_one_user` |
| (iv)  | NDCG `log2(i+2)` discount convention    | live probe of `metrics.dcg_at_k` |
| (v)   | User/item ID remapping consistency      | bounds check on every uid/iid in the splits |
| (vi)  | Popularity filtering at test time       | live simulation of one user's candidate pool |
| (vii) | @K cutoff sorting stability on ties     | `heapq.nlargest` determinism probe |

The cell raises `RuntimeError` on any **strict** failure, blocking the rest of the notebook. The previous Phase 1 audit specifically called this out as missing (Finding 3 in `Phase1_Fixed_Implementation_Audit_rev44.tex`).

In [4]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

# ---------------------------------------------------------------------------
#  rev44 Phase 1 Item #1 -- 7-point eval-protocol audit.
#
#  Self-bootstraps PROJECT_ROOT / DAMPS_DIR so this cell can run before the
#  multi-seed training cell defines those names. Re-running the audit cell
#  after the training cell has run is safe -- the values are identical.
# ---------------------------------------------------------------------------
_cwd = os.getcwd()
if _cwd.endswith(('codes', 'MMHCL_DAMPS_Project')):
    _PROJECT_ROOT = os.path.dirname(_cwd)
else:
    _PROJECT_ROOT = _cwd
_DAMPS_DIR = os.path.join(_PROJECT_ROOT, 'MMHCL_DAMPS_Project')
_DATA_DIR = os.path.join(_PROJECT_ROOT, 'data')
_PYTHON_EXE = sys.executable

# Keep the dataset name in sync with the multi-seed training cell.
_dataset = 'Clothing'

audit_script = os.path.join(_DAMPS_DIR, 'scripts', 'audit_eval_protocol.py')
if not os.path.isfile(audit_script):
    raise FileNotFoundError(
        f"Audit script not found at {audit_script}. Make sure "
        f"MMHCL_DAMPS_Project/scripts/audit_eval_protocol.py is present "
        f"(it ships with rev44 / Revision 11)."
    )

audit_cmd = [
    _PYTHON_EXE,
    audit_script,
    '--dataset', _dataset,
    '--data_path', _DATA_DIR + os.sep,
    '--core', '5',
]
print('Running rev44 Phase 1 7-point eval-protocol audit:')
print('  ' + ' '.join(audit_cmd))
print()

audit_res = subprocess.run(
    audit_cmd,
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
    cwd=_DAMPS_DIR,
)
print(audit_res.stdout)
if audit_res.returncode != 0:
    if audit_res.stderr:
        print('[stderr]')
        print(audit_res.stderr)
    raise RuntimeError(
        f"rev44 Phase 1 eval-protocol audit FAILED (exit={audit_res.returncode}). "
        f"At least one of the 7 checklist items did not pass; the four-variant "
        f"Phase 1 sweep is BLOCKED until the failure is resolved."
    )

print('[OK] rev44 Phase 1 eval-protocol audit PASSED -- proceeding to training.')

Running rev44 Phase 1 7-point eval-protocol audit:
  c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\scripts\audit_eval_protocol.py --dataset Clothing --data_path c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data\ --core 5

Auditing DAMPS-MMHCL evaluation protocol against rev44 Section 4
  cwd     = c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
  dataset = Clothing
  data_path = c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data\
  core    = 5

== (i) 5-core filtering threshold ==
  [OK]   args.core = 5 (matches MMHCL paper convention)

== (ii) Train/val/test split ratio + reproducibility ==
  [OK]   |train| = 90362 (81.0%) | |val| = 10234 (9.2%) | |test| = 10926 (9.8%)  -> to

### 3.0.5 Day 1 Diagnostic Sprint — D1 / D2 / D3

This block runs the **recall-focused, leakage-safe** probes from *Phase 1 Root-Cause Analysis and Remediation Roadmap* **§5.0** (Day 1 sprint), implemented as `MMHCL_DAMPS_Project/scripts/day1_diagnostic_sprint.py` (also summarized in `MMHCL_DAMPS_Project/README.md` §5.0).

| ID | What it checks | Hardware | When invoked |
|----|----------------|----------|--------------|
| **D1** | Metadata / APC integrity (H2): `meta_categories.npy` vs inferred `n_items`, label diversity | **CPU** | Always unless you add `--skip-d1` in a custom command |
| **D2** | Hypergraph coverage: top-*K* sweep on the Phase-1 **combined** recipe | **GPU** | `--run-d2` |
| **D3** | **Pure MMHCL** baseline (all DAMPS modules off) for the H10 sampled-eval scale check | **GPU** | `--run-d3` |
| **APC probe** | Single train with `--damps_apc 0` on the combined recipe | **GPU** | `--run-d1-apc-probe` |

**Exit code:** D1 alone may return **1** when the roadmap’s *H2 suspected* banner fires (missing meta, shape mismatch, or collapsed label diversity). Treat that as **diagnostic**, not necessarily a blocker before you run GPU probes.

**Toggles (next cell):** `DAY1_RUN_GPU = False` runs **D1 only** (cheap, no GPU training). Set `DAY1_RUN_GPU = True` to also run D2, D3, and the APC-off probe (several short full-epoch GPU jobs — lower `DAY1_DIAG_EPOCH` for smoke tests).

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys

# ---------------------------------------------------------------------------
#  Path resolution — mirror the multi-seed training cell (cwd = DAMPS_DIR).
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(("codes", "MMHCL_DAMPS_Project")):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

DAMPS_DIR: str = os.path.join(PROJECT_ROOT, "MMHCL_DAMPS_Project")
if not os.path.isdir(DAMPS_DIR):
    raise FileNotFoundError(f"Expected MMHCL_DAMPS_Project under {PROJECT_ROOT!r}")

if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
    os.chdir(DAMPS_DIR)
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)

PYTHON_EXE: str = sys.executable
DATA_PATH: str = os.path.join(PROJECT_ROOT, "data") + os.sep

dataset: str = "Clothing"

# --- toggles (see §3.0.5 markdown) ---
DAY1_RUN_GPU: bool = False
DAY1_DIAG_EPOCH: int = 250
DAY1_SEED: int = 737791071
DAY1_USE_WANDB: int = 0
DAY1_WANDB_PROJECT: str = "damps-mmhcl-clothing"
DAY1_WANDB_ENTITY: str = "baitapck51cc-uet"

cmd: list[str] = [
    PYTHON_EXE,
    os.path.join("scripts", "day1_diagnostic_sprint.py"),
    "--dataset",
    dataset,
    "--data_path",
    DATA_PATH,
    "--epoch",
    str(DAY1_DIAG_EPOCH),
    "--seed",
    str(DAY1_SEED),
    "--python_exe",
    PYTHON_EXE,
]
if DAY1_USE_WANDB:
    cmd += ["--use_wandb", "1", "--wandb_project", DAY1_WANDB_PROJECT]
    if DAY1_WANDB_ENTITY:
        cmd += ["--wandb_entity", DAY1_WANDB_ENTITY]

if DAY1_RUN_GPU:
    cmd += ["--run-d2", "--run-d3", "--run-d1-apc-probe"]

print("Day-1 diagnostic sprint")
print("  command:", subprocess.list2cmdline(cmd))
print("  cwd    :", DAMPS_DIR)

proc = subprocess.run(cmd, cwd=DAMPS_DIR, check=False)
print("\nreturn code:", proc.returncode)
if proc.returncode == 1 and not DAY1_RUN_GPU:
    print(
        "[NOTE] Exit code 1 on D1-only often means roadmap H2 suspected — "
        "see the banner above; you may still enable DAY1_RUN_GPU for D2/D3."
    )
elif proc.returncode != 0:
    print("[WARN] Non-zero exit code — review logs above.")

In [5]:
from __future__ import annotations

import os
import sys
import subprocess
import json
import re
import numpy as np
import time
import random
from pathlib import Path
from typing import Any, Optional

# ---------------------------------------------------------------------------
#  Resolve project directories. The DAMPS-MMHCL package lives under
#  MMHCL_DAMPS_Project/. ``train.py`` writes its logs relative to its own
#  cwd as ``../<dataset>/<path_name>/<path_name>.txt``, so we must launch
#  the subprocess with cwd = MMHCL_DAMPS_Project.
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(('codes', 'MMHCL_DAMPS_Project')):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

DAMPS_DIR: str = os.path.join(PROJECT_ROOT, 'MMHCL_DAMPS_Project')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')

# Switch into the DAMPS package directory so relative paths in train.py
# (``../{dataset}/...``) resolve to the workspace root.
if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
    os.chdir(DAMPS_DIR)
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)

PYTHON_EXE: str = sys.executable

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Working directory : {os.getcwd()}")
print(f"Python executable : {PYTHON_EXE}")

print("\n" + "=" * 80)
print("MULTI-SEED TRAINING: DAMPS-MMHCL — rev44 Phase 1 four-variant sweep")
print("=" * 80)

# Generate reproducible-but-fresh seeds based on current time
base_seed: int = int(time.time_ns() % (2 ** 31))
random.seed(base_seed)

# Phase 1 notebook default: 5 seeds per variant (4 × 5 = 20 runs). For a
# stricter 10-seed protocol (40 runs), set ``n_runs = 10``.
# Statistical reporting still uses paired t-tests on whatever n_runs is.
n_runs: int = 5
seeds: list[int] = [random.randint(1, 2 ** 31 - 1) for _ in range(n_runs)]

print(f"\nGenerated {n_runs} random seeds based on current time:")
print(f"Base timestamp seed : {base_seed}")
print(f"Seeds for experiments: {seeds}")
print(f"(These seeds are saved in the summary file for reproducibility)")

# W&B configuration (already created at login time, kept in sync here)
wandb_project: str = 'damps-mmhcl-clothing'
wandb_entity: str = 'baitapck51cc-uet'

# ---------------------------------------------------------------------------
#  Backbone hyper-parameters (parity with the original MMHCL paper)
# ---------------------------------------------------------------------------
dataset: str = 'Clothing'
gpu_id: int = 0
epoch: int = 250
verbose: int = 5

batch_size: int = 1024
lr: float = 0.0001
regs: float = 1e-3
embed_size: int = 64
topk: int = 5
core: int = 5
User_layers: int = 3
Item_layers: int = 2
user_loss_ratio: float = 0.03
item_loss_ratio: float = 0.07
temperature: float = 0.3       # rev44 Phase 1 anchor of the static-tau sweep ({0.2, 0.3, 0.5})
learnable_tau: int = 0         # 0 = static buffer (rev44 Phase 1 default); 1 = nn.Parameter (rev42 anchor)
clip_grad_norm: float = 1.0

# Early stopping
early_stopping_patience: int = 30
early_stopping_min_epochs: int = 75
early_stopping_min_delta: float = 0.0001
early_stopping_mode: str = 'max'
early_stopping_restore_best: int = 1

# ReduceLROnPlateau
use_reduce_lr: int = 1
reduce_lr_factor: float = 0.5
reduce_lr_patience: int = 3
reduce_lr_min: float = 1e-6

# ---------------------------------------------------------------------------
#  DAMPS-specific configuration (Revision 9, full feature set)
# ---------------------------------------------------------------------------
damps_apc: int = 1                    # Metadata-Aware Adaptive Phase Calibration
damps_avrf: int = 0                   # rev44 Phase 1 default: AVRF OFF (over-attenuates sparse Clothing); 1 = rev42 anchor
damps_imcf: int = 1                   # Residual Inter-Modal Coherence Filter
damps_permutation_fft: int = 0        # Falsifiability ablation (off in main runs)
damps_soft_routing: int = 1           # h_input = h_raw + alpha * LN(h_cal)
damps_momentum: int = 1               # Slim Momentum Encoder
damps_data_driven_prior: int = 1      # SNR-derived AVRF priors
damps_num_categories: int = 10        # Static metadata clusters for APC
damps_warmup_epochs: int = 10         # Adaptive EMA warm-up
rebuild_R: int = 5                    # Pattern B' cadence
faiss_threshold: int = 60_000         # FAISS HNSW fallback threshold
faiss_use_gpu: int = 1                # GPU FAISS resources when N >= threshold
knn_chunk_size: int = 4096            # Chunked PyTorch K-NN tile size
knn_efsearch: int = 64                # HNSW efSearch (recall/speed trade-off)
use_amp: int = 1                      # bfloat16 mixed precision

# --- Speedup Guide toggles (Speedup_Guide_complete sections 3.1 / 3.2) ---
use_torch_compile: int = 1            # Wrap DAMPS submodule in torch.compile
torch_compile_mode: str = "reduce-overhead"  # JIT mode (best for medium models)
torch_compile_dynamic: int = 1        # dynamic=True -> tolerates batch_size drift

# ---------------------------------------------------------------------------
#  rev44 / Revision 11 -- Phase 1 four-variant sweep (Section 4 of the spec)
#  Each variant overrides three knobs vs the rev42 anchor:
#     (a) anchor   : --temperature 0.1 --learnable_tau 1 --damps_avrf 1
#     (b) tau03    : --temperature 0.3 --learnable_tau 0 --damps_avrf 1
#     (c) avrf_off : --temperature 0.1 --learnable_tau 1 --damps_avrf 0
#     (d) combined : --temperature 0.3 --learnable_tau 0 --damps_avrf 0  <- recommended
# ---------------------------------------------------------------------------
phase1_variants: dict[str, dict[str, Any]] = {
    'anchor':   {'temperature': 0.1, 'learnable_tau': 1, 'damps_avrf': 1,
                 'label': '(a) rev42 anchor'},
    'tau03':    {'temperature': 0.3, 'learnable_tau': 0, 'damps_avrf': 1,
                 'label': '(b) static tau=0.3 only'},
    'avrf_off': {'temperature': 0.1, 'learnable_tau': 1, 'damps_avrf': 0,
                 'label': '(c) AVRF off only'},
    'combined': {'temperature': 0.3, 'learnable_tau': 0, 'damps_avrf': 0,
                 'label': '(d) static tau + AVRF off  (RECOMMENDED Phase 1)'},
}

# Subset to run. Default = ALL FOUR variants (5 seeds × 4 variants = 20
# training runs with the default ``n_runs``). To run ONLY the recommended
# Phase 1 config, set this to {'combined': phase1_variants['combined']}.
variants_to_run: dict[str, dict[str, Any]] = phase1_variants

# Per-variant storage: {variant_name: [per-seed result dict, ...]}
all_variant_results: dict[str, list[dict[str, Any]]] = {v: [] for v in variants_to_run}

print(f"\n{'=' * 80}")
print("rev44 PHASE 1 -- Four-Variant Training Configuration")
print(f"{'=' * 80}")
print(f"  Dataset            : {dataset}")
print(f"  GPU ID             : {gpu_id}")
print(f"  Max Epochs         : {epoch}")
print(f"  Batch Size         : {batch_size}")
print(f"  Learning Rate      : {lr}")
print(f"  Early Stopping     : patience={early_stopping_patience}, min_epochs={early_stopping_min_epochs}")
print(f"  Mixed Precision    : bfloat16  (use_amp={use_amp})")
print(f"  Rebuild Cadence    : every {rebuild_R} epochs (Pattern B')")
print(f"  DAMPS Switches     : APC={damps_apc} | IMCF={damps_imcf} | "
      f"Soft-Routing={damps_soft_routing} | Momentum={damps_momentum}")
print(f"  Variants           : {list(variants_to_run.keys())}")
print(f"  Seeds              : {seeds}")
print(f"  Total runs         : {len(variants_to_run)} variants x {n_runs} seeds = "
      f"{len(variants_to_run) * n_runs}")
print(f"  W&B Project        : {wandb_project}")


def _build_path_name(
    seed: int,
    temp_val: float,
    learn_tau: int,
    avrf_val: int,
) -> str:
    """Mirror the directory naming scheme of ``train.py::_experiment_paths``.

    Includes the ``taulearn={0,1}`` token introduced in rev44 so the four
    Phase 1 variants land in distinct log folders.
    """
    return (
        f"damps_uu_ii={User_layers}_{Item_layers}"
        f"_{user_loss_ratio}_{item_loss_ratio}"
        f"_topk={topk}_t={temp_val}_taulearn={learn_tau}_R={rebuild_R}"
        f"_apc={damps_apc}_avrf={avrf_val}_imcf={damps_imcf}"
        f"_regs={regs}_dim={embed_size}_seed={seed}_"
    )


# ---------------------------------------------------------------------------
#  Outer loop: 4 variants -> Inner loop: n_runs seeds (default 5)
# ---------------------------------------------------------------------------
total_runs: int = len(variants_to_run) * n_runs
global_run_idx: int = 0

for variant_name, ov in variants_to_run.items():
    v_temp: float = float(ov['temperature'])
    v_learn_tau: int = int(ov['learnable_tau'])
    v_avrf: int = int(ov['damps_avrf'])
    v_label: str = str(ov['label'])

    print(f"\n{'#' * 80}")
    print(f"# VARIANT '{variant_name}' -- {v_label}")
    print(f"#   --temperature {v_temp}  --learnable_tau {v_learn_tau}  --damps_avrf {v_avrf}")
    print(f"{'#' * 80}")

    for run_idx, seed in enumerate(seeds, 1):
        global_run_idx += 1
        print(f"\n{'=' * 80}")
        print(
            f"RUN {global_run_idx}/{total_runs}  (variant '{variant_name}' "
            f"seed {run_idx}/{n_runs}={seed}): Training DAMPS-MMHCL"
        )
        print(f"{'=' * 80}")

        cmd: list[str] = [
            PYTHON_EXE, 'train.py',
            # Data / schedule
            '--dataset', dataset,
            '--gpu_id', str(gpu_id),
            '--seed', str(seed),
            '--epoch', str(epoch),
            '--verbose', str(verbose),
            '--batch_size', str(batch_size),
            '--lr', str(lr),
            '--regs', str(regs),
            '--clip_grad_norm', str(clip_grad_norm),
            # Model architecture
            '--embed_size', str(embed_size),
            '--topk', str(topk),
            '--core', str(core),
            '--User_layers', str(User_layers),
            '--Item_layers', str(Item_layers),
            '--user_loss_ratio', str(user_loss_ratio),
            '--item_loss_ratio', str(item_loss_ratio),
            # ----- rev44 Phase 1 variant overrides -----
            '--temperature', str(v_temp),
            '--learnable_tau', str(v_learn_tau),
            # Early stopping
            '--early_stopping_patience', str(early_stopping_patience),
            '--early_stopping_min_epochs', str(early_stopping_min_epochs),
            '--early_stopping_min_delta', str(early_stopping_min_delta),
            '--early_stopping_mode', str(early_stopping_mode),
            '--early_stopping_restore_best', str(early_stopping_restore_best),
            # ReduceLROnPlateau
            '--use_reduce_lr', str(use_reduce_lr),
            '--reduce_lr_factor', str(reduce_lr_factor),
            '--reduce_lr_patience', str(reduce_lr_patience),
            '--reduce_lr_min', str(reduce_lr_min),
            # DAMPS-specific
            '--damps_apc', str(damps_apc),
            '--damps_avrf', str(v_avrf),                 # rev44 Phase 1 variant override
            '--damps_imcf', str(damps_imcf),
            '--damps_permutation_fft', str(damps_permutation_fft),
            '--damps_soft_routing', str(damps_soft_routing),
            '--damps_momentum', str(damps_momentum),
            '--damps_data_driven_prior', str(damps_data_driven_prior),
            '--damps_num_categories', str(damps_num_categories),
            '--damps_warmup_epochs', str(damps_warmup_epochs),
            # Pattern B' rebuild
            '--rebuild_R', str(rebuild_R),
            '--faiss_threshold', str(faiss_threshold),
            '--faiss_use_gpu', str(faiss_use_gpu),
            '--knn_chunk_size', str(knn_chunk_size),
            '--knn_efsearch', str(knn_efsearch),
            # Mixed precision
            '--use_amp', str(use_amp),
            # Speedup Guide: torch.compile on the DAMPS submodule (+25-35%)
            '--use_torch_compile', str(use_torch_compile),
            '--torch_compile_mode', str(torch_compile_mode),
            '--torch_compile_dynamic', str(torch_compile_dynamic),
            # W&B -- include the variant tag so the runs are filterable by group
            '--use_wandb', '1',
            '--wandb_project', wandb_project,
            '--wandb_entity', wandb_entity,
            '--wandb_run_name', f'phase1_{variant_name}_seed_{seed}',
            '--ablation_target', f'phase1_{variant_name}',
        ]

        print(f"Command: {' '.join(cmd)}")
        print(f"Current directory: {os.getcwd()}\n")

        env: dict[str, str] = os.environ.copy()
        env['PYTHONIOENCODING'] = 'utf-8'
        env['PYTHONUNBUFFERED'] = '1'

        result: subprocess.CompletedProcess[str] = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            env=env,
            encoding='utf-8',
            errors='replace',
        )

        if result.stdout:
            print(result.stdout)

        if result.returncode != 0:
            print(
                f"\n[WARNING] variant '{variant_name}' seed={seed} exited "
                f"with code {result.returncode}"
            )
            if result.stderr:
                print("\n[ERROR OUTPUT]:")
                print(result.stderr)
        else:
            print(
                f"\n[OK] variant '{variant_name}' seed={seed} completed successfully"
            )

        # ----- Extract metrics from the per-run log file -----------------
        path_name: str = _build_path_name(seed, v_temp, v_learn_tau, v_avrf)
        # ``ablation_target`` is appended to ``path_name`` by train.py.
        path_name = path_name + f"phase1_{variant_name}"
        log_file: str = f"../{dataset}/{path_name}/{path_name}.txt"

        if os.path.exists(log_file):
            with open(log_file, 'r', encoding='utf-8') as f:
                log_content: str = f.read()

            sep: str = r"[:=]\s*"
            # The order matters: ``BEST_Test_*`` is the canonical
            # validation-selected test result. ``BEST_Val_*`` is logged
            # alongside so the notebook can surface BOTH numbers --
            # avoiding the classic confusion where the user reads the
            # WandB ``val/recall@20`` peak (validation) and compares it
            # to the program's BEST_Test_Recall@20 (test). The two are
            # measured on different splits and should NOT be expected
            # to match.
            recall_test_match: Optional[re.Match[str]] = (
                re.search(rf'BEST_Test_Recall@20{sep}([\d.]+)', log_content)
                or re.search(rf'Test_Recall@20{sep}([\d.]+)', log_content)
            )
            recall_val_match: Optional[re.Match[str]] = re.search(
                rf'BEST_Val_Recall@20{sep}([\d.]+)', log_content
            )
            recall_val_epoch_match: Optional[re.Match[str]] = re.search(
                rf'BEST_Val_Recall_Peak_Epoch{sep}(-?\d+)', log_content
            )
            precision_match: Optional[re.Match[str]] = (
                re.search(rf'BEST_Test_Precision@20{sep}([\d.]+)', log_content)
                or re.search(rf'Test_Precision@20{sep}([\d.]+)', log_content)
            )
            ndcg_test_match: Optional[re.Match[str]] = (
                re.search(rf'BEST_Test_NDCG@20{sep}([\d.]+)', log_content)
                or re.search(rf'Test_NDCG@20{sep}([\d.]+)', log_content)
            )
            ndcg_val_match: Optional[re.Match[str]] = re.search(
                rf'BEST_Val_NDCG@20{sep}([\d.]+)', log_content
            )

            tau_match: Optional[re.Match[str]] = re.search(r'tau\s*=\s*([\d.]+)', log_content)
            sat_match: Optional[re.Match[str]] = re.search(r'tanh_sat\s*=\s*([\d.]+)', log_content)
            nnz_match: Optional[re.Match[str]] = re.search(r'nnz\s*=\s*([\d_]+)', log_content)
            avg_deg_match: Optional[re.Match[str]] = re.search(r'avg_deg\s*=\s*([\d.]+)', log_content)

            if recall_test_match and ndcg_test_match:
                precision_value: Optional[float] = (
                    float(precision_match.group(1)) if precision_match is not None else None
                )
                val_recall_value: Optional[float] = (
                    float(recall_val_match.group(1)) if recall_val_match else None
                )
                val_ndcg_value: Optional[float] = (
                    float(ndcg_val_match.group(1)) if ndcg_val_match else None
                )
                val_recall_peak_epoch: Optional[int] = (
                    int(recall_val_epoch_match.group(1))
                    if recall_val_epoch_match else None
                )
                run_result: dict[str, Any] = {
                    'variant': variant_name,
                    'variant_label': v_label,
                    'seed': seed,
                    # NOTE: keys ``recall@20`` and ``ndcg@20`` are kept for
                    # backward compatibility with the aggregator + paired-
                    # t-test cell; they map to BEST_Test_*  (i.e. the test
                    # result at the val-recall / val-ndcg peak epoch).
                    'recall@20': float(recall_test_match.group(1)),
                    'precision@20': precision_value,
                    'ndcg@20': float(ndcg_test_match.group(1)),
                    # New: surface the validation peak too so a reviewer
                    # can sanity-check the gap with the WandB val charts.
                    'val_recall@20': val_recall_value,
                    'val_ndcg@20': val_ndcg_value,
                    'val_recall_peak_epoch': val_recall_peak_epoch,
                    'tau_final': float(tau_match.group(1)) if tau_match else None,
                    'tanh_sat_final': float(sat_match.group(1)) if sat_match else None,
                    'last_nnz': int(nnz_match.group(1).replace('_', '')) if nnz_match else None,
                    'last_avg_deg': float(avg_deg_match.group(1)) if avg_deg_match else None,
                    'log_file': log_file,
                }
                all_variant_results[variant_name].append(run_result)
                precision_str: str = (
                    f"{precision_value:.6f}" if precision_value is not None else "n/a"
                )
                # Disambiguated print: explicitly say "TEST (at val-recall peak)"
                # for the headline metric and surface VAL alongside so the
                # reviewer never has to guess which split the number is on.
                print(
                    f"  Extracted TEST metrics  (snapshotted at the epoch where val_recall@20 peaked): "
                    f"Recall@20={run_result['recall@20']:.6f}, "
                    f"Precision@20={precision_str}, "
                    f"NDCG@20={run_result['ndcg@20']:.6f}"
                )
                if val_recall_value is not None or val_ndcg_value is not None:
                    epoch_str = (
                        str(val_recall_peak_epoch)
                        if val_recall_peak_epoch is not None else "?"
                    )
                    val_recall_str = (
                        f"{val_recall_value:.6f}" if val_recall_value is not None else "n/a"
                    )
                    val_ndcg_str = (
                        f"{val_ndcg_value:.6f}" if val_ndcg_value is not None else "n/a"
                    )
                    print(
                        f"  Reference VAL peaks (used for early stopping ONLY -- not the headline): "
                        f"val_Recall@20={val_recall_str} (epoch {epoch_str}), "
                        f"val_NDCG@20={val_ndcg_str}"
                    )
                if run_result['tau_final'] is not None:
                    print(f"  DAMPS diagnostics : tau={run_result['tau_final']:.4f}, "
                          f"tanh_sat={run_result['tanh_sat_final']}, "
                          f"last_nnz={run_result['last_nnz']}, "
                          f"last_avg_deg={run_result['last_avg_deg']}")
            else:
                print(f"  [WARN] Could not extract metrics from log file: {log_file}")
        else:
            print(f"  [WARN] Log file not found: {log_file}")

# ---------------------------------------------------------------------------
# Backward-compatibility alias: legacy cells (and the W&B sweep notebook)
# expect ``all_results`` to be a flat list of every per-seed run across every
# variant. We expose it here in addition to the per-variant dictionary.
# ---------------------------------------------------------------------------
all_results: list[dict[str, Any]] = [
    r for v_results in all_variant_results.values() for r in v_results
]

print(f"\n{'=' * 80}")
print("ALL DAMPS-MMHCL Phase 1 TRAINING RUNS COMPLETED")
print(f"{'=' * 80}")
print(f"Total runs collected   : {len(all_results)} / {total_runs}")
for v in variants_to_run:
    print(f"  variant '{v:<10s}' : {len(all_variant_results[v])} successful runs")

Project Root      : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing
Working directory : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
Python executable : c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe

MULTI-SEED TRAINING: DAMPS-MMHCL — rev44 Phase 1 four-variant sweep

Generated 5 random seeds based on current time:
Base timestamp seed : 2034281200
Seeds for experiments: [717692932, 737791071, 194790642, 50462948, 367290921]
(These seeds are saved in the summary file for reproducibility)

rev44 PHASE 1 -- Four-Variant Training Configuration
  Dataset            : Clothing
  GPU ID             : 0
  Max Epochs         : 250
  Batch Size         : 1024
  Learning Rate      : 0.0001
  Early Stopping     : patience=30, min_epochs=75
  Mixed Precision    : bfloat16  (use_amp=1)
  Rebuild Cadence    : every 5 epochs (Pattern B')
  DAMPS Switches     : APC=1 

## 4. Results Summary & Statistics

> **Reading this section: TEST vs VAL.** The headline numbers (`recall@20`, `precision@20`, `ndcg@20`) are computed on the **test split** and snapshotted at the epoch where `val_recall@20` (resp. `val_ndcg@20`) peaked. They match `BEST_Test_*` in the per-run text logs and `best_test_*` in the WandB run summary.
>
> The `val_recall@20` / `val_ndcg@20` columns are the **validation peaks** used only for early stopping. They match `BEST_Val_*` in the text logs and `best_val_*` in WandB, AND are exactly the maxima of the WandB `val/recall@20` / `val/ndcg@20` charts.
>
> **Validation recall/NDCG are typically higher than the test values on Amazon Clothing.** That is normal. The two are computed on different user splits — do *not* read off a chart maximum and compare it to the headline number.
>
> If you saw a peak `0.086098` in `recall@20.csv` from WandB and the program reported `0.077679`, then (a) the chart you looked at was `val/recall@20` (validation), (b) the program output was `BEST_Test_Recall@20` (test), and very likely (c) the two come from different *variants* of the four-variant sweep — every seed runs four times. To match a chart point with a headline number, check `wandb.summary.best_val_recall_peak_epoch` (added in rev44 Phase 1) and the `val_recall@20` column below.

In [6]:
from __future__ import annotations

from typing import Any
import numpy as np
import json
import os

# ---------------------------------------------------------------------------
#  Aggregate DAMPS-MMHCL multi-seed results, per Phase 1 variant.
#
#  ``all_variant_results`` is populated by the previous cell as a dict
#  keyed by variant_name (anchor / tau03 / avrf_off / combined). We emit
#  per-variant JSON summaries (consumed by the next cell's paired t-test)
#  AND a global combined summary file.
#
#  IMPORTANT -- which split each metric is computed on:
#    * ``recall@20``, ``precision@20``, ``ndcg@20``  ARE THE TEST metrics
#      (snapshotted at the epoch where val_recall@20 / val_ndcg@20 peaked).
#      They match BEST_Test_* in the per-run text logs and ``best_test_*``
#      in the WandB run summary.
#    * ``val_recall@20``, ``val_ndcg@20`` are the VALIDATION peaks (used
#      for early-stopping). They match BEST_Val_* in the text logs and
#      ``best_val_*`` in the WandB summary, AND the maxima of the
#      ``val/recall@20`` / ``val/ndcg@20`` charts in WandB.
#    * Validation recall/NDCG are typically HIGHER than the test values
#      on Amazon Clothing -- so do NOT confuse a chart maximum with the
#      headline number; the headline reports test, not val.
# ---------------------------------------------------------------------------
metrics: list[str] = ['recall@20', 'precision@20', 'ndcg@20']
val_metrics: list[str] = ['val_recall@20', 'val_ndcg@20']
diag_keys: list[str] = ['tau_final', 'tanh_sat_final', 'last_nnz', 'last_avg_deg']

# Paper anchor values (rev44 footers Section 5.2).
paper_values: dict[str, float] = {
    'recall@20': 0.0881,
    'precision@20': 0.0045,
    'ndcg@20': 0.0394,
}

# Per-variant aggregated stats {variant_name: {metric: {mean, std, values}}}
variant_stats: dict[str, dict[str, dict[str, Any]]] = {}
variant_diag_summary: dict[str, dict[str, dict[str, Any]]] = {}
variant_summary_paths: dict[str, str] = {}

if not all_variant_results or all(len(v) == 0 for v in all_variant_results.values()):
    print("\n[WARNING] No results were extracted from any DAMPS-MMHCL training run!")
    print("Please check the log files under MMHCL_DAMPS_Project/../<dataset>/ manually.")
else:
    for variant_name, run_list in all_variant_results.items():
        if not run_list:
            print(f"\n[skip] variant '{variant_name}' has no successful runs.")
            continue

        v_label: str = phase1_variants[variant_name]['label']
        print(f"\n{'=' * 80}")
        print(
            f"DAMPS-MMHCL RESULTS -- variant '{variant_name}'  {v_label}  "
            f"(n={len(run_list)})"
        )
        print(f"{'=' * 80}")
        print(f"  TEST metrics (headline -- snapshotted at val-recall / val-ndcg peak):")

        stats: dict[str, dict[str, Any]] = {}
        for metric in metrics:
            raw_values: list[Any] = [r.get(metric) for r in run_list]
            values: list[float] = [float(v) for v in raw_values if v is not None]
            if not values:
                stats[metric] = {'mean': float('nan'), 'std': float('nan'), 'values': []}
                print(f"    {metric.upper()}: no values available")
                continue
            mean_val = float(np.mean(values))
            std_val = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
            stats[metric] = {'mean': mean_val, 'std': std_val, 'values': values}
            print(
                f"    test_{metric:<14s}: mean={mean_val:.6f}  std={std_val:.6f}  "
                f"values=[{', '.join(f'{v:.4f}' for v in values)}]"
            )
        variant_stats[variant_name] = stats

        # Also surface the validation peaks used for early stopping, so
        # the reviewer can sanity-check the gap with the WandB val/* charts.
        print(f"  VAL peaks (early-stop signal -- NOT the headline):")
        for vmetric in val_metrics:
            raw_v: list[Any] = [r.get(vmetric) for r in run_list]
            v_values: list[float] = [float(v) for v in raw_v if v is not None]
            if not v_values:
                print(f"    {vmetric.upper():18s}: no values available")
                continue
            v_mean = float(np.mean(v_values))
            v_std = float(np.std(v_values, ddof=1)) if len(v_values) > 1 else 0.0
            stats[vmetric] = {'mean': v_mean, 'std': v_std, 'values': v_values}
            print(
                f"    {vmetric:<18s}: mean={v_mean:.6f}  std={v_std:.6f}  "
                f"values=[{', '.join(f'{v:.4f}' for v in v_values)}]"
            )

        # Per-variant DAMPS diagnostics
        diag_summary: dict[str, dict[str, Any]] = {}
        for k in diag_keys:
            raw_d: list[Any] = [r.get(k) for r in run_list]
            clean: list[float] = [float(v) for v in raw_d if v is not None]
            if clean:
                diag_summary[k] = {
                    'mean': float(np.mean(clean)),
                    'std':  float(np.std(clean, ddof=1)) if len(clean) > 1 else 0.0,
                    'values': clean,
                }
        variant_diag_summary[variant_name] = diag_summary
        if diag_summary:
            for k, vd in diag_summary.items():
                print(
                    f"  diag {k:18s}: mean={vd['mean']:.4f}  std={vd['std']:.4f}"
                )

        # Comparison with paper anchor
        print(f"  ----- vs MMHCL paper (Recall@20=0.0881, NDCG@20=0.0394) -----")
        for metric in metrics:
            if not stats[metric]['values']:
                continue
            ours = stats[metric]['mean']
            pv = paper_values[metric]
            pct = (ours - pv) / pv * 100.0
            status = "[OK]" if abs(pct) < 10 else "[CHECK]"
            print(f"    {metric.upper():12s}: ours={ours:.4f}  paper={pv:.4f}  ({pct:+.1f}%) {status}")

        # ----- Persist per-variant JSON summary (consumed by t-test cell) ----
        ov = phase1_variants[variant_name]
        v_summary_path: str = f"../{dataset}/phase1_{variant_name}_summary.json"
        v_summary_data: dict[str, Any] = {
            'variant': variant_name,
            'label': v_label,
            'overrides': {
                'temperature': float(ov['temperature']),
                'learnable_tau': int(ov['learnable_tau']),
                'damps_avrf': int(ov['damps_avrf']),
            },
            'n_runs': len(run_list),
            'model': 'DAMPS-MMHCL (rev44 / Revision 11 -- Phase 1)',
            'seeds': [r['seed'] for r in run_list],
            'statistics': {
                m: {
                    'mean': float(stats[m]['mean']),
                    'std':  float(stats[m]['std']),
                    'values': list(stats[m]['values']),
                } for m in metrics
            },
            'damps_diagnostics': diag_summary,
            'individual_results': run_list,
        }
        with open(v_summary_path, 'w', encoding='utf-8') as f:
            json.dump(v_summary_data, f, indent=2)
        variant_summary_paths[variant_name] = v_summary_path
        print(f"  [OK] per-variant JSON saved -> {v_summary_path}")

    # ---- Global combined summary (legacy file name kept for back-compat) ---
    print(f"\n{'=' * 80}")
    print("rev44 PHASE 1 -- ALL VARIANTS (mean +/- std over seeds per variant)")
    print(f"{'=' * 80}")
    print(
        "  Showing test_*  (headline) AND val_*  (early-stop signal). "
        "These are different splits;"
    )
    print(
        "  do NOT compare a WandB val/recall@20 chart maximum to the headline test_recall@20."
    )
    print()
    print(
        f"  {'Variant':<40s} "
        f"{'test_Recall@20':>16s} {'test_NDCG@20':>14s}  ||  "
        f"{'val_Recall@20':>14s} {'val_NDCG@20':>14s}"
    )
    for variant_name in variants_to_run:
        if variant_name not in variant_stats:
            continue
        s = variant_stats[variant_name]
        rl = phase1_variants[variant_name]['label']
        t_rec = f"{s['recall@20']['mean']:.4f}+-{s['recall@20']['std']:.4f}"
        t_ndc = f"{s['ndcg@20']['mean']:.4f}+-{s['ndcg@20']['std']:.4f}"
        v_rec_s = s.get('val_recall@20')
        v_ndc_s = s.get('val_ndcg@20')
        v_rec = (
            f"{v_rec_s['mean']:.4f}+-{v_rec_s['std']:.4f}"
            if v_rec_s and v_rec_s['values'] else " n/a"
        )
        v_ndc = (
            f"{v_ndc_s['mean']:.4f}+-{v_ndc_s['std']:.4f}"
            if v_ndc_s and v_ndc_s['values'] else " n/a"
        )
        print(
            f"  {rl:<40s} {t_rec:>16s} {t_ndc:>14s}  ||  {v_rec:>14s} {v_ndc:>14s}"
        )

    summary_file: str = f"../{dataset}/multi_seed_summary.json"
    combined_data: dict[str, Any] = {
        'model': 'DAMPS-MMHCL (rev44 / Revision 11 -- Phase 1)',
        'config': {
            'batch_size': batch_size,
            'max_epochs': epoch,
            'lr': lr,
            'embed_size': embed_size,
            'topk': topk,
            'rebuild_R': rebuild_R,
            'use_amp': bool(use_amp),
            'damps_apc': bool(damps_apc),
            'damps_imcf': bool(damps_imcf),
            'damps_soft_routing': bool(damps_soft_routing),
            'damps_momentum': bool(damps_momentum),
            'damps_data_driven_prior': bool(damps_data_driven_prior),
            'damps_num_categories': damps_num_categories,
            'damps_warmup_epochs': damps_warmup_epochs,
            'early_stopping_patience': early_stopping_patience,
            'early_stopping_min_epochs': early_stopping_min_epochs,
            'wandb_project': wandb_project,
        },
        'phase1_variants': phase1_variants,
        'variant_summary_paths': variant_summary_paths,
        'variant_statistics': {
            v: {
                m: {
                    'mean': float(variant_stats[v][m]['mean']),
                    'std':  float(variant_stats[v][m]['std']),
                } for m in metrics
            } for v in variant_stats
        },
        'variant_damps_diagnostics': variant_diag_summary,
        'all_individual_results': all_results,
    }
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(combined_data, f, indent=2)
    print(f"\n[OK] combined JSON summary saved -> {summary_file}")

    # ---- Human-readable summary (kept for legacy consumers) --------------
    summary_txt_file: str = f"../{dataset}/multi_seed_summary.txt"
    with open(summary_txt_file, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write(
            f"DAMPS-MMHCL rev44 Phase 1 -- 4 variants x {n_runs} seeds "
            f"= {sum(len(r) for r in all_variant_results.values())} runs\n"
        )
        f.write(
            f"Configuration: batch_size={batch_size}, max_epochs={epoch}, "
            f"lr={lr}, rebuild_R={rebuild_R}, use_amp={bool(use_amp)}\n"
        )
        f.write("=" * 80 + "\n\n")
        for variant_name in variants_to_run:
            if variant_name not in variant_stats:
                f.write(f"variant '{variant_name}': no successful runs\n\n")
                continue
            s = variant_stats[variant_name]
            f.write(
                f"variant '{variant_name}'  ({phase1_variants[variant_name]['label']}):\n"
            )
            for m in metrics:
                if not s[m]['values']:
                    continue
                f.write(
                    f"  {m.upper():14s}: mean={s[m]['mean']:.6f}  "
                    f"std={s[m]['std']:.6f}  "
                    f"values={[f'{v:.6f}' for v in s[m]['values']]}\n"
                )
            f.write("\n")
    print(f"[OK] human-readable summary saved -> {summary_txt_file}")

    # Legacy variable expected by the Excel-export cell. We expose the
    # COMBINED stats (mean/std across the recommended variant 'combined'
    # if available, else across all runs) under the old name.
    if 'combined' in variant_stats:
        stats: dict[str, dict[str, Any]] = variant_stats['combined']
        diag_summary = variant_diag_summary.get('combined', {})
    else:
        # Fall back to flat aggregation when only one variant ran.
        stats = {}
        for m in metrics:
            v_all: list[float] = []
            for vstats in variant_stats.values():
                v_all.extend(vstats[m]['values'])
            if v_all:
                stats[m] = {
                    'mean': float(np.mean(v_all)),
                    'std':  float(np.std(v_all, ddof=1)) if len(v_all) > 1 else 0.0,
                    'values': v_all,
                }
            else:
                stats[m] = {'mean': float('nan'), 'std': float('nan'), 'values': []}
        diag_summary = {}


DAMPS-MMHCL RESULTS -- variant 'anchor'  (a) rev42 anchor  (n=5)
  TEST metrics (headline -- snapshotted at val-recall / val-ndcg peak):
    test_recall@20     : mean=0.079493  std=0.002483  values=[0.0827, 0.0814, 0.0766, 0.0782, 0.0786]
    test_precision@20  : mean=0.004427  std=0.000129  values=[0.0046, 0.0045, 0.0043, 0.0044, 0.0044]
    test_ndcg@20       : mean=0.037981  std=0.000897  values=[0.0393, 0.0381, 0.0380, 0.0369, 0.0376]
  VAL peaks (early-stop signal -- NOT the headline):
    val_recall@20     : mean=0.080485  std=0.001898  values=[0.0805, 0.0785, 0.0834, 0.0809, 0.0792]
    val_ndcg@20       : mean=0.037141  std=0.000626  values=[0.0372, 0.0364, 0.0380, 0.0375, 0.0367]
  diag tau_final         : mean=1.0000  std=0.0000
  diag last_avg_deg      : mean=69.2960  std=0.7635
  ----- vs MMHCL paper (Recall@20=0.0881, NDCG@20=0.0394) -----
    RECALL@20   : ours=0.0795  paper=0.0881  (-9.8%) [OK]
    PRECISION@20: ours=0.0044  paper=0.0045  (-1.6%) [OK]
    NDCG@20     : 

## 4.5 rev44 Phase 1 — Bonferroni-corrected paired *t*-test + stop-gate

This cell consumes the per-variant JSON summaries written by the previous cell (`../{dataset}/phase1_<variant>_summary.json`) and:

1. Runs **paired *t*-tests** (matched seeds) for each non-anchor variant — `tau03`, `avrf_off`, `combined` — against the rev42 anchor `anchor`, on both `Recall@20` and `NDCG@20`.
2. Applies a **Bonferroni correction** with `N = #non-anchor variants` (typically 3) so the family-wise α stays at 0.05.
3. Reports the **rev44 §5.2 quantitative stop-gate verdict** per variant:
   * `Recall@20 ≥ 0.0900` → **PAPER VALIDATED** (strictly beats MMHCL paper's 0.0881).
   * `Recall@20 ≥ 0.0870 ∧ NDCG@20 ≥ 0.0390` → **PHASE 1 PASS** (closes coverage gap; advance to Phase 2).
   * Else → **PHASE 1 FAIL** — re-audit the eval protocol (checklist (iii) all-ranking vs sampled is the prime suspect).

The cell is a thin wrapper around `MMHCL_DAMPS_Project/scripts/paired_ttest.py` with `--bonferroni N`. Skip it without affecting downstream cells if you only ran a single variant.

In [7]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np

# ---------------------------------------------------------------------------
#  rev44 / Revision 11 -- Phase 1 Bonferroni-corrected paired t-test cell.
#
#  Pulls per-variant aggregated JSON summaries written by the previous cell,
#  runs paired t-tests vs the anchor with --bonferroni N (where N = # of
#  non-anchor variants present), and prints the rev44 Section 5.2 stop-gate
#  verdict per variant.
# ---------------------------------------------------------------------------

# Make MMHCL_DAMPS_Project/scripts/ importable so we can call the helper
# directly without spawning a subprocess (faster + cleaner traceback).
_DAMPS_DIR = Path(DAMPS_DIR) if 'DAMPS_DIR' in dir() else Path(os.getcwd())
if str(_DAMPS_DIR) not in sys.path:
    sys.path.insert(0, str(_DAMPS_DIR))

try:
    from scripts.paired_ttest import paired_ttest_report
except Exception as exc:                                            # pragma: no cover
    print(f"[ERROR] failed to import scripts.paired_ttest: {exc}")
    print("        Is MMHCL_DAMPS_Project/scripts/paired_ttest.py present?")
    raise

# Stop-gate thresholds (rev44 §5.2).
ACCEPTANCE_RECALL: float = 0.0870
ACCEPTANCE_NDCG: float = 0.0390
PAPER_VALIDATION_RECALL: float = 0.0900

ANCHOR_NAME: str = 'anchor'

# Resolve where cell 14 wrote its per-variant summaries
ds_dir: Path = Path(f"../{dataset}").resolve()


def _load_variant(name: str) -> dict[str, Any] | None:
    p = ds_dir / f"phase1_{name}_summary.json"
    if not p.is_file():
        return None
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)


anchor_data: dict[str, Any] | None = _load_variant(ANCHOR_NAME)
if anchor_data is None or not anchor_data['statistics']['recall@20']['values']:
    print(
        f"\n[SKIP] Anchor variant '{ANCHOR_NAME}' has no aggregated "
        f"summary at {ds_dir / 'phase1_anchor_summary.json'}; cannot run "
        f"paired t-tests. (Did you run the training cell with the anchor "
        f"variant included?)"
    )
else:
    other_variants: list[str] = [
        v for v in variants_to_run
        if v != ANCHOR_NAME and v in all_variant_results
        and len(all_variant_results[v]) > 0
    ]
    bonferroni_N: int = max(1, len(other_variants))

    print(f"\n{'=' * 80}")
    print(
        f"rev44 Phase 1 -- Paired t-tests vs '{ANCHOR_NAME}' "
        f"(Bonferroni correction = {bonferroni_N})"
    )
    print(f"{'=' * 80}")

    ttest_report: dict[str, dict[str, dict[str, Any]]] = {}
    for v_name in other_variants:
        v_data = _load_variant(v_name)
        if v_data is None:
            print(f"[skip] missing summary for variant '{v_name}'")
            continue
        ttest_report[v_name] = {}
        for metric in ('recall@20', 'ndcg@20'):
            a_vals: list[float] = list(anchor_data['statistics'][metric]['values'])
            v_vals: list[float] = list(v_data['statistics'][metric]['values'])
            n = min(len(a_vals), len(v_vals))
            if n < 2:
                print(
                    f"[skip] variant '{v_name}' vs anchor on {metric}: "
                    f"only {n} paired observations"
                )
                continue
            print(f"\n----- variant '{v_name}' vs '{ANCHOR_NAME}' on {metric} (n={n}) -----")
            r = paired_ttest_report(
                damps_scores=v_vals[:n],
                baseline_scores=a_vals[:n],
                alpha=0.05,
                bonferroni=bonferroni_N,
                label_a=v_data.get('label', v_name),
                label_b=anchor_data.get('label', ANCHOR_NAME),
            )
            ttest_report[v_name][metric] = r

    # ----- rev44 Section 5.2 stop-gate verdict -----
    print(f"\n{'=' * 80}")
    print("rev44 Section 5.2 stop-gate verdict (per variant)")
    print(f"{'=' * 80}")

    final_verdicts: dict[str, dict[str, Any]] = {}
    for v_name in variants_to_run:
        v_data = _load_variant(v_name)
        if v_data is None or not v_data['statistics']['recall@20']['values']:
            print(f"  variant '{v_name:<10s}' -- no data")
            continue
        rmean = float(v_data['statistics']['recall@20']['mean'])
        nmean = float(v_data['statistics']['ndcg@20']['mean'])
        beat_paper = rmean >= PAPER_VALIDATION_RECALL
        beat_acceptance = (
            rmean >= ACCEPTANCE_RECALL and nmean >= ACCEPTANCE_NDCG
        )
        if beat_paper:
            verdict = "PAPER VALIDATED  (Recall@20 >= 0.0900 -- strictly beats MMHCL paper)"
        elif beat_acceptance:
            verdict = "PHASE 1 PASS     (Recall@20 >= 0.0870 AND NDCG@20 >= 0.0390)"
        else:
            verdict = (
                "PHASE 1 FAIL     (re-audit eval protocol; checklist (iii) "
                "all-ranking vs sampled is the highest-suspicion mismatch)"
            )
        final_verdicts[v_name] = {
            'recall@20_mean': rmean,
            'ndcg@20_mean': nmean,
            'beat_acceptance': bool(beat_acceptance),
            'beat_paper': bool(beat_paper),
            'verdict': verdict,
        }
        print(
            f"  variant '{v_name:<10s}'  Recall@20={rmean:.4f}  "
            f"NDCG@20={nmean:.4f} -> {verdict}"
        )

    # Save the consolidated stop-gate + t-test report so reviewers can
    # inspect it without re-running the notebook.
    final_report_path: Path = ds_dir / "phase1_stopgate_report.json"
    with final_report_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                'stopgate_thresholds': {
                    'acceptance_recall@20': ACCEPTANCE_RECALL,
                    'acceptance_ndcg@20':   ACCEPTANCE_NDCG,
                    'paper_validation_recall@20': PAPER_VALIDATION_RECALL,
                },
                'bonferroni_N': bonferroni_N,
                'paired_ttest': ttest_report,
                'verdicts': final_verdicts,
            },
            f,
            indent=2,
        )
    print(f"\n[OK] consolidated stop-gate report -> {final_report_path}")


rev44 Phase 1 -- Paired t-tests vs 'anchor' (Bonferroni correction = 3)

----- variant 'tau03' vs 'anchor' on recall@20 (n=5) -----
|  Method            |   Mean   |    Std   |   N  |
|--------------------|----------|----------|------|
|  (b) static tau=0.3 only|  0.0829  |  0.0007  |  5   |
|  (a) rev42 anchor  |  0.0795  |  0.0025  |  5   |

Paired t-test : t=2.674, p=0.05557 [n.s.]
Bonferroni(3) -> corrected alpha = 0.05/3 = 0.0166667
98% CI of mean diff = [-0.0016, 0.0085]  (Bonferroni-widened)
alpha = 0.05, alpha_eff = 0.0166667, n = 5

----- variant 'tau03' vs 'anchor' on ndcg@20 (n=5) -----
|  Method            |   Mean   |    Std   |   N  |
|--------------------|----------|----------|------|
|  (b) static tau=0.3 only|  0.0397  |  0.0005  |  5   |
|  (a) rev42 anchor  |  0.0380  |  0.0009  |  5   |

Paired t-test : t=4.306, p=0.01258 [significant]
Bonferroni(3) -> corrected alpha = 0.05/3 = 0.0166667
98% CI of mean diff = [0.0001, 0.0033]  (Bonferroni-widened)
alpha = 0.05, al

## 5. Export Results to Excel

In [8]:
from __future__ import annotations

from typing import Any

import pandas as pd
from openpyxl.styles import Font, PatternFill

if len(all_results) > 0:
    print("=" * 80)
    print("EXPORTING DAMPS-MMHCL RESULTS TO EXCEL")
    print("=" * 80)

    excel_file: str = f"../{dataset}/DAMPS_MMHCL_{dataset}_Results_Local.xlsx"

    # ---- Individual Results (incl. DAMPS diagnostics) -----------------------
    individual_data: list[dict[str, Any]] = []
    for i, r in enumerate(all_results, 1):
        row: dict[str, Any] = {
            'Run': i,
            'Seed': r['seed'],
            'Recall@20': r['recall@20'],
            'Precision@20': r.get('precision@20'),
            'NDCG@20': r['ndcg@20'],
            'tau_final': r.get('tau_final'),
            'tanh_sat_final': r.get('tanh_sat_final'),
            'last_nnz': r.get('last_nnz'),
            'last_avg_deg': r.get('last_avg_deg'),
        }
        individual_data.append(row)

    df_individual: pd.DataFrame = pd.DataFrame(individual_data)

    # ---- Summary statistics --------------------------------------------------
    summary_data_list: list[dict[str, Any]] = []
    for metric in metrics:
        if not stats[metric]['values']:
            continue
        summary_data_list.append({
            'Metric': metric.upper(),
            'Mean': stats[metric]['mean'],
            'Std': stats[metric]['std'],
            'Mean +/- Std': f"{stats[metric]['mean']:.6f} +/- {stats[metric]['std']:.6f}",
        })
    df_summary: pd.DataFrame = pd.DataFrame(summary_data_list)

    # ---- Paper comparison ----------------------------------------------------
    comparison_data: list[dict[str, Any]] = []
    for metric in metrics:
        if not stats[metric]['values']:
            continue
        our_mean: float = stats[metric]['mean']
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100.0
        comparison_data.append({
            'Metric': metric.upper(),
            'DAMPS-MMHCL (Ours)': our_mean,
            'MMHCL Paper': paper_val,
            'Difference (%)': diff_pct,
        })
    df_comparison: pd.DataFrame = pd.DataFrame(comparison_data)

    # ---- DAMPS configuration / diagnostics ----------------------------------
    config_rows: list[dict[str, Any]] = [
        {'Setting': 'Model', 'Value': 'DAMPS-MMHCL (Revision 9)'},
        {'Setting': 'Dataset', 'Value': dataset},
        {'Setting': 'Batch size', 'Value': batch_size},
        {'Setting': 'Max epochs', 'Value': epoch},
        {'Setting': 'Learning rate', 'Value': lr},
        {'Setting': 'Embedding dim', 'Value': embed_size},
        {'Setting': 'Top-k (KNN)', 'Value': topk},
        {'Setting': 'Temperature init', 'Value': temperature},
        {'Setting': 'Rebuild R (Pattern B\')', 'Value': rebuild_R},
        {'Setting': 'use_amp (bf16)', 'Value': bool(use_amp)},
        {'Setting': 'damps_apc', 'Value': bool(damps_apc)},
        {'Setting': 'damps_avrf', 'Value': bool(damps_avrf)},
        {'Setting': 'damps_imcf', 'Value': bool(damps_imcf)},
        {'Setting': 'damps_soft_routing', 'Value': bool(damps_soft_routing)},
        {'Setting': 'damps_momentum', 'Value': bool(damps_momentum)},
        {'Setting': 'damps_data_driven_prior', 'Value': bool(damps_data_driven_prior)},
        {'Setting': 'damps_num_categories', 'Value': damps_num_categories},
        {'Setting': 'damps_warmup_epochs', 'Value': damps_warmup_epochs},
        {'Setting': 'early_stopping_patience', 'Value': early_stopping_patience},
        {'Setting': 'early_stopping_min_epochs', 'Value': early_stopping_min_epochs},
        {'Setting': 'W&B project', 'Value': wandb_project},
    ]
    df_config: pd.DataFrame = pd.DataFrame(config_rows)

    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        df_individual.to_excel(writer, sheet_name='Individual Results', index=False)
        df_summary.to_excel(writer, sheet_name='Summary Statistics', index=False)
        df_comparison.to_excel(writer, sheet_name='Paper Comparison', index=False)
        df_config.to_excel(writer, sheet_name='DAMPS Config', index=False)

        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            for col in ws.columns:
                max_length: int = 0
                column: str = col[0].column_letter
                for cell in col:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except Exception:
                        pass
                ws.column_dimensions[column].width = max_length + 2

            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='E0E0E0', end_color='E0E0E0', fill_type='solid')

    print(f"\n[OK] Results exported to: {excel_file}")
    print(f"  - Sheet 1: Individual Results ({len(all_results)} runs)")
    print(f"  - Sheet 2: Summary Statistics")
    print(f"  - Sheet 3: Paper Comparison")
    print(f"  - Sheet 4: DAMPS Config")
else:
    print("No results to export.")

EXPORTING DAMPS-MMHCL RESULTS TO EXCEL

[OK] Results exported to: ../Clothing/DAMPS_MMHCL_Clothing_Results_Local.xlsx
  - Sheet 1: Individual Results (20 runs)
  - Sheet 2: Summary Statistics
  - Sheet 3: Paper Comparison
  - Sheet 4: DAMPS Config


## 5.5 Permutation-FFT Falsifiability Ablation (Optional)

This is the falsifiability test from **Specification Section 6, Item 8** (and compliance check **INFO 4** of `DAMPS_compliance_check_report.tex`). It re-runs the full DAMPS-MMHCL pipeline twice per seed:

- **Arm A** — `--damps_permutation_fft 0` (standard 1-D rFFT, the default)
- **Arm B** — `--damps_permutation_fft 1` (input is permuted with a fixed random permutation before FFT)

After both arms have completed, a paired t-test is performed on Recall@20 and NDCG@20. The spec's fallback rule:

| `|gap|` (Recall@20) | Decision |
|---|---|
| `< 1 %` | Switch the spectral basis to **DCT-II** (the 1-D FFT axis is not informative). |
| `>= 1 %` | The standard 1-D FFT is **validated** and remains the default. |

Skip this section unless you are preparing the final paper / camera-ready ablation table — each pair adds two full training runs.

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess

# Toggle to True when you want to actually run the falsifiability test.
RUN_PERMUTATION_FFT_ABLATION: bool = False

if not RUN_PERMUTATION_FFT_ABLATION:
    print("[SKIP] Permutation-FFT ablation disabled (set RUN_PERMUTATION_FFT_ABLATION = True to run).")
else:
    # ``DAMPS_DIR`` was defined in the environment-setup cell (cell 2) and
    # is also the cwd that ``train.py`` expects (so its ``../{dataset}/...``
    # paths resolve to the workspace root).
    if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
        os.chdir(DAMPS_DIR)
    if DAMPS_DIR not in sys.path:
        sys.path.insert(0, DAMPS_DIR)

    PYTHON_EXE: str = sys.executable
    perm_seeds: list[int] = [42, 43, 44]   # 3 paired seeds is enough for INFO 4
    perm_dataset: str = 'Clothing'
    perm_epoch: int = 250

    common_flags: list[str] = [
        '--dataset', perm_dataset,
        '--epoch', str(perm_epoch),
        '--damps_apc', '1', '--damps_avrf', '1', '--damps_imcf', '1',
        '--damps_soft_routing', '1', '--damps_momentum', '1',
        '--damps_data_driven_prior', '1', '--use_amp', '1',
    ]

    for seed in perm_seeds:
        for perm_on, tag_prefix in [(0, 'perm_fft_off'), (1, 'perm_fft_on')]:
            tag: str = f"{tag_prefix}_seed{seed}"
            print("=" * 70)
            print(f"[ablation] {tag}")
            print("=" * 70)
            cmd: list[str] = [
                PYTHON_EXE, 'train.py',
                '--seed', str(seed),
                '--ablation_target', tag,
                '--damps_permutation_fft', str(perm_on),
            ] + common_flags
            env = os.environ.copy()
            env['PYTHONIOENCODING'] = 'utf-8'
            env['PYTHONUNBUFFERED'] = '1'
            r = subprocess.run(
                cmd, capture_output=True, text=True, env=env,
                encoding='utf-8', errors='replace',
            )
            if r.stdout:
                print(r.stdout[-2000:])
            if r.returncode != 0:
                print(f"[FAIL] {tag} exited with code {r.returncode}")
                if r.stderr:
                    print(r.stderr[-2000:])

    # ----- Aggregate the paired runs and run the t-test --------------------
    aggregator: str = os.path.join(DAMPS_DIR, 'scripts', '_aggregate_permutation_fft.py')
    if os.path.exists(aggregator):
        print("=" * 70)
        print("Aggregating Permutation-FFT runs and running paired t-test...")
        print("=" * 70)
        agg_cmd: list[str] = [
            PYTHON_EXE, aggregator,
            '--dataset', perm_dataset,
            '--seeds', *[str(s) for s in perm_seeds],
        ]
        env = os.environ.copy()
        env['PYTHONIOENCODING'] = 'utf-8'
        r = subprocess.run(agg_cmd, capture_output=True, text=True, env=env,
                           encoding='utf-8', errors='replace', cwd=DAMPS_DIR)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    else:
        print(f"[WARN] Aggregator script not found at {aggregator}.")

## 6. Check W&B Run Status (Optional)

In [ ]:
from __future__ import annotations

import wandb

print(f"Checking DAMPS-MMHCL runs for {wandb_entity}/{wandb_project} ...")

try:
    api: wandb.Api = wandb.Api()
    runs: wandb.apis.public.Runs = api.runs(f"{wandb_entity}/{wandb_project}")

    if len(runs) == 0:
        print("No runs found in this project yet.")
    else:
        print(f"\nFound {len(runs)} runs. Latest 5 status:")
        print("-" * 70)
        for run in runs[:5]:
            print(f"Run Name : {run.name}")
            print(f"Status   : {run.state}")
            print(f"Created  : {run.created_at}")
            # Standard MMHCL/DAMPS keys logged by ``train.py``
            for key in (
                'val/recall@20',
                'val/ndcg@20',
                'best_test_recall@20',
                'best_test_precision@20',
                'best_test_ndcg@20',
                'damps/tau',
                'damps/tanh_sat',
                'rebuild/nnz',
                'rebuild/avg_deg',
            ):
                if key in run.summary:
                    val = run.summary[key]
                    try:
                        print(f"  {key:28s}: {float(val):.4f}")
                    except (TypeError, ValueError):
                        print(f"  {key:28s}: {val}")
            print("-" * 70)
except Exception as e:
    print(f"Error fetching run status: {e}")
    print("Make sure W&B is logged in correctly.")

## Troubleshooting (DAMPS-MMHCL)

### Common Issues

1. **CUDA Out of Memory** — even with bf16 AMP enabled
   ```python
   batch_size = 512   # or 256
   knn_chunk_size = 2048  # tile the KNN distance matrix more aggressively
   ```
   You can also disable the Slim Momentum Encoder (`damps_momentum=0`) to free up to ~2 GB.

2. **`bfloat16 not supported`** — pre-Ampere GPUs cannot run the bf16 path
   ```python
   use_amp = 0   # falls back to fp32
   ```

3. **`faiss-gpu` not installed** — only required for `N >= 60_000` items
   - Default Clothing dataset has 23 033 items, so the chunked PyTorch path is used automatically.
   - Install via `pip install faiss-gpu` (Linux) or `conda install -c pytorch faiss-gpu` if you switch to a larger dataset.

4. **W&B login issues** — run `wandb login` in a terminal first

5. **Missing modules** — re-run the *Install Dependencies* cell after restarting the kernel

6. **Data not found** — ensure the multi-modal feature files exist under
   `data/Clothing/5-core/` (UI_mat_sym.pth, User_mat_rw.pth, image_feats.npy, text_feats.npy, …).

7. **Permutation-FFT ablation** — to run the falsifiability check:
   ```python
   damps_permutation_fft = 1
   ```
   The DAMPS-MMHCL spec predicts a measurable performance drop when this is enabled.

### DAMPS Diagnostic Sanity-Checks (look for these in logs)
- `tanh_sat` should stabilise in `[0.1, 0.6]` after warm-up — values near 1.0 mean AVRF logits are saturated.
- `nnz` and `avg_deg` per rebuild should plateau, not grow unboundedly (Pattern B' is working).
- `tau` (learnable InfoNCE temperature) starts at `0.1` (spec Section 3.1) and is allowed to drift; healthy runs keep it in roughly `[0.05, 0.30]` after warm-up.

### Expected Results
- **Original MMHCL (paper)** — Recall@20 = 0.0881, Precision@20 = 0.0045, NDCG@20 = 0.0394
- **DAMPS-MMHCL** — see the JSON / Excel summary produced by Step 4 / Step 5.